# Basic 02 - Semantic Normalization
Small, linear flow for semantic mapping, strict controls, and validation diagnostics.


## 1) Imports


In [19]:
from pathlib import Path
import sys
import warnings
import logging

import pandas as pd
from IPython.display import display


## 2) Quiet logs (optional)


In [20]:
warnings.filterwarnings("ignore")
logging.getLogger("isa_phm").setLevel(logging.ERROR)


## 3) Make local package importable


In [21]:
def _ensure_local_package() -> None:
    cwd = Path.cwd().resolve()
    search_roots = [cwd, *cwd.parents]
    for root in search_roots:
        if (root / "isa_phm").is_dir() and (root / "pyproject.toml").exists():
            root_s = str(root)
            if root_s not in sys.path:
                sys.path.insert(0, root_s)
            return
    raise RuntimeError("Could not locate python-wrapper root with isa_phm package.")

_ensure_local_package()


## 4) Import wrapper + errors


In [22]:
from isa_phm import ISAWrapper
from isa_phm.errors import ValidationError


## 5) Pick ISA JSON


In [23]:
ISA_JSON = Path(r"g:/ISA/ISA-PHM-Wizard/src/tests/fixtures/golden/isa-phm-out-milling.json")
DATA_ROOT = ISA_JSON.parent

print("ISA-JSON :", ISA_JSON)
print("DATA_ROOT:", DATA_ROOT)
print("Exists   :", ISA_JSON.exists())


ISA-JSON : g:\ISA\ISA-PHM-Wizard\src\tests\fixtures\golden\isa-phm-out-milling.json
DATA_ROOT: g:\ISA\ISA-PHM-Wizard\src\tests\fixtures\golden
Exists   : True


## 6) Optional semantic override config path


In [24]:
# Keep None if you don't use a project-specific override config.
SEMANTIC_OVERRIDE = None
# Example:
# SEMANTIC_OVERRIDE = Path(r"g:/ISA/my-semantic-overrides.json")
SEMANTIC_OVERRIDE


## 7) Build wrapper


In [25]:
wrapper = ISAWrapper(
    ISA_JSON,
    data_root=DATA_ROOT,
    strict_validation=False,
    semantic_config_path=SEMANTIC_OVERRIDE,
)


## 8) Build semantic manifest


In [26]:
manifest = wrapper.semantic_manifest()


## 9) Manifest diagnostics


In [27]:
display(pd.DataFrame([manifest.diagnostics.model_dump()]))


,total_fields,mapped_fields,unknown_fields,ambiguous_fields,unknown_ratio,ambiguous_ratio,missing_override_fields,strict_violations
0,1072,176,896,0,0.835821,0.0,"[Amplifier, Amplifier max load, Amplifier min ...",[SEM_UNKNOWN_RATIO_EXCEEDED]


## 10) Semantic factors for first study


In [28]:
study = wrapper.study(wrapper.list_studies()[0].title)
sem_factors = study.semantic_factors()
sem_factors_df = pd.DataFrame([f.model_dump() for f in sem_factors])

display(sem_factors_df[["source_name", "semantic_key", "status", "confidence", "provenance"]])


,source_name,semantic_key,status,confidence,provenance
0,VB,damage_vb,mapped,0.99,built_in_exact
1,Cutting Speed,operating_speed,mapped,0.99,built_in_exact
2,Depth of Cut,depth_of_cut,mapped,0.99,built_in_exact
3,Feed,feed_rate,mapped,0.99,built_in_exact
4,Material,material,mapped,0.99,built_in_exact


## 11) Semantic parameters for first assay


In [29]:
assay = study.assay(study.list_assays()[0].assay_id)
sem_params = assay.semantic_parameters()
meas_df = pd.DataFrame([p.model_dump() for p in sem_params["measurement"]])
proc_df = pd.DataFrame([p.model_dump() for p in sem_params["processing"]])


## 12) Measurement parameter mappings


In [30]:
display(meas_df[["source_name", "semantic_key", "status", "confidence", "provenance"]])


,source_name,semantic_key,status,confidence,provenance
0,Sensor Location,unknown,unknown,0.0,no_match
1,Sensor Frequency Range,unknown,unknown,0.0,no_match
2,Filter Type,unknown,unknown,0.0,no_match
3,HP cutoff frequency,unknown,unknown,0.0,no_match
4,Amplifier,unknown,unknown,0.0,no_match
5,Amplifier max load,unknown,unknown,0.0,no_match
6,Amplifier min load,unknown,unknown,0.0,no_match
7,Amplifier sensitivity,unknown,unknown,0.0,no_match
8,Amplifier output,unknown,unknown,0.0,no_match


## 13) Processing parameter mappings


In [31]:
display(proc_df[["source_name", "semantic_key", "status", "confidence", "provenance"]])


,source_name,semantic_key,status,confidence,provenance
0,Filter type,unknown,unknown,0.0000,no_match
1,Filter roll-off rate,unknown,unknown,0.0000,no_match
2,Filter LP cutoff frequency,filter_cutoff_frequency,mapped,0.9388,built_in_fuzzy
3,Filter HP cutoff frequency,filter_cutoff_frequency,mapped,0.9388,built_in_fuzzy
4,Smoothening method,unknown,unknown,0.0000,no_match
5,Smoothening time constant,unknown,unknown,0.0000,no_match
6,Smoothening sampling frequency,unknown,unknown,0.0000,no_match


## 14) Strict semantic check example


In [32]:
try:
    strict_manifest = wrapper.semantic_manifest(
        strict=True,
        max_unknown_ratio=0.05,
        max_ambiguous_ratio=0.0,
        require_override_config=False,
    )
    print("Strict semantic validation passed.")
    display(pd.DataFrame([strict_manifest.diagnostics.model_dump()]))
except ValidationError as exc:
    print("Strict semantic validation failed:")
    print(exc)


Strict semantic validation failed:
Semantic strict validation failed (codes=['SEM_UNKNOWN_RATIO_EXCEEDED'], unknown_ratio=0.8358, ambiguous_ratio=0.0000, missing_override_fields=['Amplifier', 'Amplifier max load', 'Amplifier min load', 'Amplifier output', 'Amplifier sensitivity', 'Current converter', 'Current type', 'Filter Type', 'Filter roll-off rate', 'Filter type', 'HP cutoff frequency', 'Power supply', 'Power supply output', 'Preamplifier', 'Sensor Frequency Range', 'Sensor Location', 'Smoothening method', 'Smoothening sampling frequency', 'Smoothening time constant']).


## 15) Dataset validation report (semantic + metadata checks)


In [33]:
report = wrapper.validate_dataset(
    check_files=True,
    semantic_strict=False,
    max_unknown_ratio=0.05,
    max_ambiguous_ratio=0.0,
    require_override_config=False,
)
summary = {
    "ok": report.ok,
    "n_errors": report.n_errors,
    "n_warnings": report.n_warnings,
    "n_info": report.n_info,
}
display(pd.DataFrame([summary]))


,ok,n_errors,n_warnings,n_info
0,True,0,4,1


## 16) First validation issues


In [34]:
issues_df = pd.DataFrame([i.model_dump() for i in report.issues])
display(issues_df.head(20))


,code,level,scope,message,context
0,CONTACT_MISSING_EMAIL,info,multiple,Contact has no email address.,"{'n_occurrences': 2, 'scopes_sample': ['contac..."
1,FILE_NOT_FOUND,warning,multiple,raw data file path does not exist on disk.,"{'file_type': 'raw', 'path': 'D:\DPL\ISA3\Exam..."
2,FILE_NOT_FOUND,warning,multiple,processed data file path does not exist on disk.,"{'file_type': 'processed', 'path': 'D:\DPL\ISA..."
3,SEM_UNKNOWN_FIELDS,warning,semantic,Unknown semantic fields detected.,"{'unknown_fields': 896, 'unknown_ratio': 0.835..."
4,SEM_UNKNOWN_RATIO_EXCEEDED,warning,semantic,Semantic strict threshold would fail.,"{'max_unknown_ratio': 0.05, 'max_ambiguous_rat..."


## 17) AI context export with semantic + validation sections


In [35]:
ai_ctx = wrapper.ai_context(include_semantics=True, include_validation=True)
list(ai_ctx.keys())


['schema_version',
 'generated_at_utc',
 'source_path',
 'investigation',
 'contacts',
 'publications',
 'studies',
 'assays',
 'factors',
 'semantic_manifest',
 'validation_report']

## 18) AI semantic diagnostics snippet


In [36]:
display(pd.DataFrame([ai_ctx["semantic_manifest"]["diagnostics"]]))


,total_fields,mapped_fields,unknown_fields,ambiguous_fields,unknown_ratio,ambiguous_ratio,missing_override_fields,strict_violations
0,1072,176,896,0,0.835821,0.0,"[Amplifier, Amplifier max load, Amplifier min ...",[SEM_UNKNOWN_RATIO_EXCEEDED]
